In [ ]:
# ============================================================
# STAGE 2: ADVANCED TEXT PREPROCESSING
# CELL 1: IMPORTS AND SETUP
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path
import json
import re
import html
import hashlib
import unicodedata
import warnings

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

# Project paths
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "dataset").exists():
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "dataset").exists():
    PROJECT_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Could not find the dataset folder. "
        "Check your project directory."
    )

DATASET_DIR = PROJECT_DIR / "dataset"
PREPROCESSED_DIR = PROJECT_DIR / "preprocessed_data"
REPORT_DIR = PROJECT_DIR / "preprocessing_reports"

PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load datasets if not already defined in globals
if "datasets" not in globals():
    FILES = {
        "train": "darkreddit_train.csv",
        "validation": "darkreddit_val.csv",
        "test": "darkreddit_test.csv"
    }
    datasets = {}
    for split_name, filename in FILES.items():
        file_path = DATASET_DIR / filename
        if not file_path.exists():
            raise FileNotFoundError(f"Dataset file not found: {file_path}")
        df = pd.read_csv(
            file_path,
            usecols=["text1", "text2"],
            encoding="utf-8"
        )
        datasets[split_name] = df

required_columns = {"text1", "text2"}

for split_name, df in datasets.items():
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{split_name} is missing columns: {missing_columns}"
        )

print("Stage 2 setup complete.")
print("Project directory:", PROJECT_DIR)
print("Available splits:", list(datasets.keys()))
for split_name, df in datasets.items():
    print(f"  {split_name}: {df.shape}")
